# CI/CD for ML — prototyping the quality gate

The pipeline pieces live in `../src/train.py`, `../tests/test_model.py` and
`../workflow-example.yml`. This notebook prototypes the model-quality gate logic
before you wire it into GitHub Actions.

## 1. Train a small model and record baseline accuracy

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

def train(**params):
    model = LogisticRegression(max_iter=200, **params).fit(X_train, y_train)
    return model, model.score(X_test, y_test)

baseline_model, baseline_acc = train()
print(f"baseline accuracy: {baseline_acc:.4f}")

## 2. Define the gate and assert against it

In [ ]:
MIN_ACCURACY = 0.85  # the fixed floor — same value as ../tests/test_model.py

assert baseline_acc >= MIN_ACCURACY, f"{baseline_acc:.4f} below gate {MIN_ACCURACY}"
print(f"GATE PASSED: {baseline_acc:.4f} >= {MIN_ACCURACY}")

## 3. Simulate a regression and confirm the gate catches it

In [ ]:
# A deliberately bad config: heavy regularization strangles the model.
bad_model, bad_acc = train(C=0.0001)
print(f"regressed accuracy: {bad_acc:.4f}")

try:
    assert bad_acc >= MIN_ACCURACY, f"{bad_acc:.4f} below gate {MIN_ACCURACY}"
except AssertionError as e:
    print("GATE FAILED (as intended) ->", e)

## 4. A relative gate: candidate vs. current baseline

In [ ]:
TOLERANCE = 0.01  # candidate may be at most 1 point worse than baseline

def relative_gate(candidate_acc, baseline_acc, tolerance=TOLERANCE):
    ok = candidate_acc >= baseline_acc - tolerance
    verdict = "PASS" if ok else "FAIL"
    print(f"[{verdict}] candidate={candidate_acc:.4f} vs baseline={baseline_acc:.4f} (tol {tolerance})")
    return ok

candidate_model, candidate_acc = train(C=10.0)
relative_gate(candidate_acc, baseline_acc)
relative_gate(bad_acc, baseline_acc)

## 5. Mini-project notes: wiring it into CI

- Run the real gate locally: `cd .. && pip install -r requirements.txt && pytest`
- Copy `../workflow-example.yml` to `.github/workflows/ci.yml` at the repo root to activate it.
- Decide: fixed floor, relative gate, or both? Note your reasoning here.
- Watch out for nondeterminism: pin `random_state` everywhere, or gate with a small margin.